# D6-TV1 — Adapter screening trên Colab GPU

Repo `haduckien-raccoon/msila`, branch `g2/member1`. **ViT-B/16**, E2: frozen DINOv3 → một Adapter → Decoder.
Grid `r=[64,128,256]`, `d=[256,512,768]`: **9 cặp × 8 category = 72 run**. Mỗi run có Adapter, Decoder và optimizer mới; chỉ backbone frozen được dùng lại trong một process. Không chạy E2 main hoặc D7–D11.

Chọn Runtime → Change runtime type → GPU (T4/L4/A100). Chạy các phần 1–3 trước, rồi bật cell full screening ở phần 4. Mở notebook không tự chạy jobs. Snapshot thực nghiệm ban đầu: **NOT RUN, 0/72; chưa có selection lock**.

Notebook chỉ điều phối `scripts/g2_colab_screening.py`, gọi API của `scripts/run_g2.py` và trainer `src/train/g2_e2.py`. Không có training loop hoặc AU-PRO thứ hai trong notebook. Cần commit/push các file đi kèm notebook lên branch trước khi clone trên Colab; setup kiểm tra API mới và ghi commit thực tế.

## 1. Setup — Drive, Git commit, dependencies, dữ liệu, checkpoint

Chỉnh INPUT trong cell dưới và OUTPUT trong cell riêng tiếp theo. Dataset/weights được copy từ Drive sang `/content`; `.tar.gz` được giải nén trên disk Colab. Chỉ lấy TRAIN/good và VALIDATION/good, giữ nguyên pixel native; TEST không được giải nén hoặc dùng chọn r/d.

In [ ]:
# @title INPUT + mount Google Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
from pathlib import Path
import os, sys, subprocess, json, hashlib, shutil, tarfile, re, signal, csv, html
from IPython.display import display, Markdown, HTML, Image
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'

# INPUT: đổi theo layout Drive của bạn.
PROJECT_DRIVE = Path('/content/drive/MyDrive/[Q3-4] 2026/[S7] Computer Vision/CV-Nhóm 9')
ARCHIVE_DIR = PROJECT_DRIVE / 'data'
ARCHIVE_PATTERNS = ['*.tar.gz']
CHECKPOINT_ON_DRIVE = PROJECT_DRIVE / 'weights/dinov3_vitb16_pretrain_lvd1689m-73cec8be.pth'
GITHUB_URL = 'https://github.com/haduckien-raccoon/msila.git'
GITHUB_REF = 'g2/member1'
GIT_COMMIT = None  # None: HEAD của branch lúc setup; hoặc SHA đầy đủ để tái lập.
DINO_URL = 'https://github.com/facebookresearch/dinov3.git'
DINO_COMMIT = '6876159a11b4df116f30f667f8c9888617df0751'
LOCAL_WORK = Path('/content/msila_g2')  # Giữ nguyên path khi resume.
REPO_DIR = LOCAL_WORK / 'repo'
DINO_DIR = LOCAL_WORK / 'dinov3'
VENV_DIR = LOCAL_WORK / 'venv'
LOCAL_ARCHIVES = LOCAL_WORK / 'archives'
LOCAL_DATA = LOCAL_WORK / 'dataset'
LOCAL_CHECKPOINT = LOCAL_WORK / 'weights' / CHECKPOINT_ON_DRIVE.name
CONFIG_DIR = LOCAL_WORK / 'config'
RUNNER_CONFIG = CONFIG_DIR / 'g2_runner.yaml'
CATEGORIES = ['can', 'fabric', 'fruit_jelly', 'rice', 'sheet_metal', 'vial', 'wallplugs', 'walnuts']

# None: tự đo khả năng GPU, chọn batch chung trước khi bắt đầu study.
# 4: giữ batch của contract G2. Batch lớn làm đổi số updates/epoch: dùng RUN_ID mới.
BATCH_SIZE = None
MAX_BATCH_SIZE = 128
GPU_MEMORY_FRACTION = 0.80  # Giữ khoảng 20% VRAM dự phòng, đo bằng training thật.
NUM_WORKERS = 2
MAX_DEV_SOURCES = 4  # Default runner hiện có. None: toàn bộ VALIDATION/good, phải chọn trước study.
CHECKPOINT_INTERVAL_STEPS = 20

# Phần 4: all/all là 72 jobs; hoặc ['rice', 'can'] và ['64:256', '128:512'].
RUN_CATEGORIES = ['all']
RUN_PAIRS = ['all']
RUN_FULL_SCREENING = False  # Chỉ bật True sau khi preflight và smoke PASS.
for path in (LOCAL_WORK, LOCAL_ARCHIVES, LOCAL_DATA, LOCAL_CHECKPOINT.parent, CONFIG_DIR):
    path.mkdir(parents=True, exist_ok=True)


In [ ]:
# @title OUTPUT — chỉnh layout Drive tại đây
# OUTPUT: tất cả đường dẫn bền vững nằm trong cell này.
RUN_ID = 'vitb16_d6_seed2026_v1'
DRIVE_OUTPUT_BASE = PROJECT_DRIVE / 'outputs/G2/D6'
DRIVE_OUTPUT_ROOT = DRIVE_OUTPUT_BASE / RUN_ID
LOCAL_OUTPUT_ROOT = LOCAL_WORK / 'outputs' / RUN_ID
DRIVE_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
LOCAL_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print('Drive:', DRIVE_OUTPUT_ROOT)
print('Run tree: dinov3_vitb16/full/adapter_screen/adapter_rR_dD/CATEGORY/')
print('Reports: dinov3_vitb16/full/colab_report/')


In [ ]:
# @title GPU + Git commit + dependencies (safetensors>=0.8)
import torch
if not torch.cuda.is_available():
    raise RuntimeError('NOT RUN: chọn GPU T4/L4/A100 trong Runtime settings.')
if tuple(int(v) for v in torch.__version__.split('+')[0].split('.')[:2]) < (2, 6):
    raise RuntimeError(f'Cần CUDA PyTorch>=2.6 của Colab; đang có {torch.__version__}.')
props = torch.cuda.get_device_properties(0)
print('GPU:', props.name, 'VRAM GiB:', round(props.total_memory / 2**30, 1))

def checkout_pinned(url, directory, ref):
    if not (directory / '.git').exists():
        subprocess.run(['git', 'clone', '--depth', '1', url, str(directory)], check=True)
    dirty = subprocess.check_output(['git', '-C', str(directory), 'status', '--porcelain'], text=True)
    if dirty.strip():
        raise RuntimeError(f'Repo có chỉnh sửa local chưa lưu: {directory}. Giữ repo Colab sạch để khóa commit.')
    subprocess.run(['git', '-C', str(directory), 'fetch', '--depth', '1', 'origin', ref], check=True)
    commit = subprocess.check_output(['git', '-C', str(directory), 'rev-parse', 'FETCH_HEAD'], text=True).strip()
    subprocess.run(['git', '-C', str(directory), 'checkout', '--detach', commit], check=True)
    return commit

RESOLVED_COMMIT = checkout_pinned(GITHUB_URL, REPO_DIR, GIT_COMMIT or GITHUB_REF)
RESOLVED_DINO_COMMIT = checkout_pinned(DINO_URL, DINO_DIR, DINO_COMMIT)
previous_setup = DRIVE_OUTPUT_ROOT / 'setup_manifest.json'
if previous_setup.is_file() and json.loads(previous_setup.read_text())['commit'] != RESOLVED_COMMIT:
    raise RuntimeError('RUN_ID đã khóa Git commit khác. Đặt GIT_COMMIT bằng commit trong manifest để resume, hoặc RUN_ID mới.')
if not (REPO_DIR / 'scripts/g2_colab_screening.py').is_file():
    raise RuntimeError('Commit chưa có helper Colab. Commit/push các file D6-TV1 mới lên g2/member1 rồi chạy lại setup.')
print('MS-ILA commit:', RESOLVED_COMMIT, 'DINOv3 commit:', RESOLVED_DINO_COMMIT)

# Venv kế thừa CUDA torch; dependencies chạy trong process mới, không đổi kernel Colab.
if not (VENV_DIR / 'bin/python').is_file():
    subprocess.run([sys.executable, '-m', 'venv', '--system-site-packages', '--without-pip', str(VENV_DIR)], check=True)
PYTHON = str(VENV_DIR / 'bin/python')  # Giữ symlink trong venv, không resolve().
requirements = [line for line in (REPO_DIR / 'requirements.txt').read_text().splitlines()
                if line.strip() and not line.lstrip().startswith(('#', 'torch'))]
if 'safetensors>=0.8' not in requirements:
    requirements.append('safetensors>=0.8')
dependency_file = LOCAL_WORK / 'requirements_colab.txt'
dependency_file.write_text('\n'.join(requirements) + '\n')
subprocess.run([PYTHON, '-m', 'pip', 'install', '-q', '-r', str(dependency_file)], check=True)
TRAIN_ENV = dict(os.environ, PYTHONUNBUFFERED='1', PYTEST_DISABLE_PLUGIN_AUTOLOAD='1',
                 CUBLAS_WORKSPACE_CONFIG=':4096:8', MPLCONFIGDIR=str(LOCAL_WORK / 'mpl'),
                 TMPDIR=str(LOCAL_WORK / 'tmp'))
Path(TRAIN_ENV['TMPDIR']).mkdir(parents=True, exist_ok=True)
check_script = """
import torch, safetensors
from packaging.version import Version
assert torch.cuda.is_available()
assert Version(safetensors.__version__) >= Version('0.8.0')
print('Training:', torch.__version__, torch.cuda.get_device_name(0), 'safetensors', safetensors.__version__)
"""
subprocess.run([PYTHON, '-c', check_script], env=TRAIN_ENV, check=True)


In [ ]:
# @title Copy .tar.gz + checkpoint vào /content, giải nén good local
def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(8 * 1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

def copy_atomic(source, target):
    source, target = Path(source), Path(target)
    target.parent.mkdir(parents=True, exist_ok=True)
    temporary = target.with_name(target.name + '.copying')
    shutil.copy2(source, temporary)
    os.replace(temporary, target)

def good_target(member_name, archive_name):
    relative = Path(member_name)
    if relative.is_absolute() or '..' in relative.parts or '\\' in member_name:
        raise ValueError(f'Unsafe archive path: {member_name}')
    parts = relative.parts
    lower = [part.lower() for part in parts]
    if any(part.startswith('test') or part == 'ground_truth' for part in lower):
        return None
    cats = [cat for cat in CATEGORIES if cat in lower]
    if not cats:
        cats = [cat for cat in CATEGORIES if re.search(rf'(^|[^a-z0-9]){cat}([^a-z0-9]|$)', archive_name.lower())]
    if len(cats) != 1:
        return None
    for index, part in enumerate(lower[:-2]):
        if part in ('train', 'validation') and lower[index + 1] == 'good':
            tail = parts[index + 2:]
            if (Path(tail[-1]).suffix.lower() in ('.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff')
                    and not any('mask' in item.lower() or 'ground_truth' in item.lower() for item in tail)):
                return Path(cats[0]) / part.upper() / 'good' / Path(*tail)
    return None

archives = sorted({path for pattern in ARCHIVE_PATTERNS for path in ARCHIVE_DIR.glob(pattern)})
if not archives:
    raise FileNotFoundError(f'Không tìm thấy .tar.gz: {ARCHIVE_DIR}, {ARCHIVE_PATTERNS}')
if not CHECKPOINT_ON_DRIVE.is_file():
    raise FileNotFoundError(CHECKPOINT_ON_DRIVE)
required_bytes = sum(path.stat().st_size for path in archives if not (LOCAL_ARCHIVES / path.name).is_file())
required_bytes += 0 if LOCAL_CHECKPOINT.is_file() else CHECKPOINT_ON_DRIVE.stat().st_size
required_bytes += 4 * 2**30
if shutil.disk_usage(LOCAL_WORK).free < required_bytes:
    raise RuntimeError('Không đủ disk local để copy archives/weights và giữ khoảng trống cho giải nén/checkpoints.')
archive_manifest = []
for source in archives:
    target = LOCAL_ARCHIVES / source.name
    digest = sha256_file(source)
    if not target.is_file() or sha256_file(target) != digest:
        copy_atomic(source, target)
    if sha256_file(target) != digest:
        raise ValueError(f'Archive copy checksum mismatch: {source.name}')
    archive_manifest.append(dict(name=source.name, sha256=digest, bytes=source.stat().st_size))
    print('LOCAL archive:', target.name)
# Tất cả archives đã về Colab trước khi giải nén.
extracted = 0
for local_archive in [LOCAL_ARCHIVES / row['name'] for row in archive_manifest]:
    with tarfile.open(local_archive, 'r:gz') as archive:
        for member in archive:
            if not member.isfile():
                continue
            relative = good_target(member.name, local_archive.name)
            if relative is None:
                continue
            target = LOCAL_DATA / relative
            if not target.resolve().is_relative_to(LOCAL_DATA.resolve()):
                raise ValueError('Extraction escaped LOCAL_DATA')
            target.parent.mkdir(parents=True, exist_ok=True)
            if shutil.disk_usage(LOCAL_WORK).free < member.size + 2**30:
                raise RuntimeError('Không đủ disk local để giải nén ảnh native và giữ dự phòng.')
            temporary = target.with_name(target.name + '.extracting')
            with archive.extractfile(member) as src, temporary.open('wb') as dest:
                shutil.copyfileobj(src, dest)
            if target.exists() and sha256_file(target) != sha256_file(temporary):
                temporary.unlink()
                raise ValueError(f'Conflicting archive image: {target}; kiểm tra input/LOCAL_WORK.')
            os.replace(temporary, target)
            extracted += 1
print('TRAIN/VALIDATION good images extracted:', extracted)
weights_sha = sha256_file(CHECKPOINT_ON_DRIVE)
if not LOCAL_CHECKPOINT.is_file() or sha256_file(LOCAL_CHECKPOINT) != weights_sha:
    copy_atomic(CHECKPOINT_ON_DRIVE, LOCAL_CHECKPOINT)
if sha256_file(LOCAL_CHECKPOINT) != weights_sha:
    raise ValueError('DINOv3 checkpoint copy checksum mismatch')
setup_manifest = dict(commit=RESOLVED_COMMIT, dino_commit=RESOLVED_DINO_COMMIT,
                      archives=archive_manifest, checkpoint_sha256=weights_sha,
                      categories=CATEGORIES, extracted_good_images=extracted, test_extracted=False)
(LOCAL_WORK / 'setup_manifest.json').write_text(json.dumps(setup_manifest, indent=2))
if previous_setup.is_file() and json.loads(previous_setup.read_text()) != setup_manifest:
    raise RuntimeError('Inputs/archives/checkpoint đã đổi trong RUN_ID cũ. Chọn RUN_ID mới; manifest cũ được giữ nguyên.')
copy_atomic(LOCAL_WORK / 'setup_manifest.json', DRIVE_OUTPUT_ROOT / 'setup_manifest.json')


In [ ]:
# @title Config chung + helper orchestration (không có train loop)
settings = dict(repo=str(REPO_DIR), dino=str(DINO_DIR), data=str(LOCAL_DATA), weights=str(LOCAL_CHECKPOINT),
                config=str(CONFIG_DIR), workers=NUM_WORKERS, dev_sources=MAX_DEV_SOURCES,
                checkpoint_interval=CHECKPOINT_INTERVAL_STEPS)
config_script = """
import json, sys
from pathlib import Path
from src.train.screen_representation import read_yaml, save_yaml
s = json.loads(sys.argv[1]); root = Path(s['repo']); folder = Path(s['config'])
cfg = read_yaml(root / 'configs/g2_experiments.yaml')
cfg['backbone'].update(name='dinov3_vitb16', repo_dir=s['dino'], weights=s['weights'])
cfg['data'].update(root=s['data'], num_workers=s['workers'], max_dev_sources=s['dev_sources'])
cfg['synthetic_protocol'] = str(root / 'configs/full_scale_synthetic.yaml')
save_yaml(cfg, folder / 'g2_experiments.yaml')
runner = read_yaml(root / 'configs/g2_runner.yaml')
runner.update(backbone='dinov3_vitb16', model_config=str(folder / 'g2_experiments.yaml'))
runner['training']['checkpoint_interval_steps'] = s['checkpoint_interval']
runner['smoke']['epochs'] = 2  # Đủ 2 updates kể cả batch lớn chỉ có 1 batch/epoch.
save_yaml(runner, folder / 'g2_runner.yaml')
"""
subprocess.run([PYTHON, '-c', config_script, json.dumps(settings)], cwd=REPO_DIR, env=TRAIN_ENV, check=True)
for source, name in ((CONFIG_DIR / 'g2_experiments.yaml', 'input_g2_experiments.yaml'), (RUNNER_CONFIG, 'input_g2_runner.yaml')):
    destination = DRIVE_OUTPUT_ROOT / name
    if destination.is_file() and destination.read_text() != source.read_text():
        raise RuntimeError('Config đã đổi trong RUN_ID cũ; dùng RUN_ID mới để giữ study có thể tái lập.')
    copy_atomic(source, destination)

def run_screening_tool(action, categories=None, pairs=None):
    command = [PYTHON, str(REPO_DIR / 'scripts/g2_colab_screening.py'), '--action', action,
               '--config', str(RUNNER_CONFIG), '--local-root', str(LOCAL_OUTPUT_ROOT),
               '--drive-root', str(DRIVE_OUTPUT_ROOT), '--device', 'cuda',
               '--max-batch', str(MAX_BATCH_SIZE), '--memory-fraction', str(GPU_MEMORY_FRACTION)]
    if BATCH_SIZE is not None:
        command += ['--batch-size', str(BATCH_SIZE)]
    if categories is not None:
        command += ['--categories', *categories]
    if pairs is not None:
        command += ['--pairs', *pairs]
    console_dir = LOCAL_OUTPUT_ROOT / '_session'
    console_dir.mkdir(parents=True, exist_ok=True)
    console_path = console_dir / f'{action}.log'
    process = subprocess.Popen(command, cwd=REPO_DIR, env=TRAIN_ENV, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True, bufsize=1)
    try:
        with console_path.open('a') as console:
            for line in process.stdout:
                print(line, end=''); console.write(line); console.flush()
                match = re.search(r'epoch=\d+ step=(\d+)/', line)
                if match and int(match[1]) % CHECKPOINT_INTERVAL_STEPS == 0:
                    copy_atomic(console_path, DRIVE_OUTPUT_ROOT / '_session' / console_path.name)
        returncode = process.wait()
    except BaseException:
        process.send_signal(signal.SIGINT)
        try: process.wait(timeout=30)
        except subprocess.TimeoutExpired:
            process.terminate()
            try: process.wait(timeout=10)
            except subprocess.TimeoutExpired: process.kill(); process.wait()
        raise
    finally:
        copy_atomic(console_path, DRIVE_OUTPUT_ROOT / '_session' / console_path.name)
    if returncode:
        raise RuntimeError(f'{action} exit={returncode}. Xem log; kết quả/checkpoint đã có được backup theo run.')


## 2. Preflight — kiểm tra trước screening

Kiểm tra đủ 8 category, TRAIN/DEV không trùng source, checkpoint ViT-B đúng kiến trúc, seed/config/grid, frozen backbone và CUDA. Capacity probe dùng **TRAIN tiles và E2 thật**, chạy hai optimizer steps cho cặp lớn nhất `(256,768)`; kết quả probe không được tính vào PASS/72.

Auto batch thử mức tối đa theo VRAM: khoảng T4 ≤16, L4 ≤32, A100 40 GB ≤64, A100 80 GB ≤128; đây là **trần thử**, không phải batch bảo đảm fit. Nếu CUDA OOM hoặc không còn 20% dự phòng, thử batch nhỏ hơn. Batch cuối và evaluation tile batch được lưu `colab_batch_profile.json`, dùng chung toàn bộ 72 run. FP32, AMP/TF32 tắt, tile 512, loss/seed/split/synthetic giữ config hiện có. Batch lớn thay đổi số update trong 20 epochs; bảng budget từng category được in trước khi train. Đặt `BATCH_SIZE=4` để dùng batch của contract.

Resume trên GPU khác giữ batch cũ và probe lại khả năng fit; không tự giảm batch giữa các run. Muốn tăng batch trên L4/A100, dùng `RUN_ID` mới. Không có cam kết tránh mọi OOM: nếu bộ nhớ thay đổi sau preflight, run dừng và giữ checkpoint đã backup.

In [ ]:
# @title Preflight + auto batch, khóa cấu hình trước mọi run
run_screening_tool('preflight')


## 3. Smoke — một cặp × một category

Chạy đúng `rice`, `(r,d)=(64,256)`, 2 optimizer steps theo smoke budget của runner; cho phép tối đa 2 epochs để batch lớn vẫn có đủ 2 updates. Kiểm tra loss hữu hạn, forward/backward, backbone bất biến, Adapter/Decoder cập nhật, native synthetic DEV AU-PRO hợp lệ và best/last checkpoint có checksum. Output `smoke/` tách riêng, luôn **0 đóng góp vào selection**. Full screening yêu cầu smoke này hợp lệ.

In [ ]:
# @title Smoke GPU: rice / r64 d256
run_screening_tool('smoke', categories=['rice'], pairs=['64:256'])
smoke_acceptance = DRIVE_OUTPUT_ROOT / 'dinov3_vitb16/smoke/adapter_screen/adapter_r64_d256/rice/smoke_acceptance.json'
display(json.loads(smoke_acceptance.read_text()))


## 4. Full screening — 72 run hoặc một nhóm cần chạy

Chỉ cell này gọi full screening. Đặt `RUN_FULL_SCREENING=True`; mặc định INPUT `RUN_CATEGORIES=['all']`, `RUN_PAIRS=['all']` chạy 72 jobs. Có thể chọn `['rice', 'can']` và `['64:256', '128:512']`, hoặc từng nhóm 3 cặp. Các nhóm cùng `RUN_ID` vẫn giữ grid 9 cặp, config, batch và budget đã khóa.

Completed run chỉ skip sau khi TV1 xác minh budget, config, DEV, checkpoint và checksum. Run chưa hoàn thành resume từ `last.pt` với optimizer/RNG/cursor; PASS hỏng bị BLOCKED. Backbone chỉ load một lần cho mỗi lần gọi helper, mọi run khởi tạo head/optimizer mới. Checkpoints backup sau mỗi lần runner lưu (chu kỳ 20 step, cuối epoch); log/metric/config được sync sau từng run. Khi bấm Stop, helper cố sync qua `finally`; nếu runtime bị thu hồi đột ngột, resume từ checkpoint gần nhất trên Drive.

Không dùng TEST để chọn. Chỉ tạo `adapter_selection_lock.json` khi TV1 xác minh đủ **72/72 real full CUDA runs**; chọn macro mean trên đủ 8 category, tie-break: ít Adapter trainable parameters → r nhỏ hơn → d nhỏ hơn. Checkpoint trong run chọn DEV AU-PRO tốt nhất, exact tie lấy epoch sớm nhất.

In [ ]:
# @title FULL SCREENING — mặc định tắt, chạy riêng cell này
if RUN_FULL_SCREENING:
    run_screening_tool('screen', categories=RUN_CATEGORIES, pairs=RUN_PAIRS)
else:
    print('Full screening NOT RUN. Bật RUN_FULL_SCREENING=True rồi chạy riêng cell này khi sẵn sàng.')


## 5. Tổng hợp và xuất kết quả nghiệm thu

Bảng 72 hàng theo category/r/d và bảng 9 macro được cập nhật trên Drive sau mỗi run. Hàng chưa hợp lệ giữ metric trống; ô heatmap xám biểu thị thiếu category. Không chọn cặp tối ưu trên phần kết quả chưa đủ 72. Report cuối đọc lại chứng cứ và checksum qua selector hiện có.

Output bền vững: `setup_manifest.json`, input YAML, session logs; dưới `dinov3_vitb16/`: batch profile, `run_manifest.json`, `preflight.json`, protocol lock, mỗi run gồm config/hash/log/metrics/best.pt/last.pt + SHA256. `full/colab_report/` gồm `screening_72.csv/json`, `macro_9.csv/json`, `acceptance.json`, `rd_heatmap.png`. Selection lock nằm ở `full/adapter_selection_lock.json` sau khi đủ điều kiện.

Nếu Colab ngắt: giữ `RUN_ID`, commit (`GIT_COMMIT` lấy từ manifest), DINO commit, đường dẫn local và scientific settings; chạy lại Setup → Preflight → Smoke → Full screening. Profile và artifacts được restore từ Drive, nhóm đã hoàn tất sẽ được skip. Dataset/weights mới hoặc batch/budget/commit khác cần `RUN_ID` mới.

Tham khảo: [Colab: copy archive và giải nén local](https://research.google.com/colaboratory/faq.html#drive-quota), [DINOv3 official](https://github.com/facebookresearch/dinov3). Code acceptance CPU không thay thế nghiệm thu thực nghiệm Colab: **chưa chạy Colab = NOT RUN, 0/72**.

In [ ]:
# @title Validate lại 72 kết quả + macro/heatmap/selection lock
run_screening_tool('report')
report_dir = DRIVE_OUTPUT_ROOT / 'dinov3_vitb16/full/colab_report'
acceptance = json.loads((report_dir / 'acceptance.json').read_text())
print(f"{acceptance['status']}: {acceptance['valid_runs']}/72")
print('Selected pair:', acceptance['selected_pair'])
print('Tie-break:', acceptance['selection_rule']['tie_break'])
print('Lock:', acceptance['selection_lock'])

def display_csv(path):
    with path.open() as stream:
        rows = list(csv.DictReader(stream))
    keys = list(rows[0])
    markup = '<table><thead><tr>' + ''.join(f'<th>{html.escape(k)}</th>' for k in keys) + '</tr></thead><tbody>'
    markup += ''.join('<tr>' + ''.join(f'<td>{html.escape(row[k])}</td>' for k in keys) + '</tr>' for row in rows)
    display(HTML(markup + '</tbody></table>'))
display_csv(report_dir / 'screening_72.csv')
display_csv(report_dir / 'macro_9.csv')
display(Image(filename=str(report_dir / 'rd_heatmap.png')))
